# Avance 5. Modelo final: ensambles y evaluación en datos no vistos

**Inferencia de propiedades físicas de galaxias a partir de imágenes multibanda** · Instituto Tecnológico y de Estudios Superiores de Monterrey · Maestría en Inteligencia Artificial Aplicada · **Equipo 24** · Colaboración con el Instituto de Astronomía de la UNAM

| Integrante | Matrícula |
|---|---|
| Luz Copelia Minutti Pérez | A01796921 |
| Erick Alberto Bustos Cruz | A01378966 |
| José Florencio Maguey Peralta | A01796727 |

**Asesores:** Dr. Iván Reyes Amezcua (ITESM) · Dr. Héctor Hernández Toledo y Dr. José Antonio Vázquez Mata (IA-UNAM)

**Fase de CRISP-ML(Q):** 3, *Machine Learning Model Engineering* (Studer et al. 2021). **Avance 5:** modelo final.

## 1. Planteamiento

En el Avance 4 comparamos seis modelos individuales y vimos que ninguno domina en todas las propiedades: la ResNet50 con redshift estima mejor la masa, y DINOv3 con LoRA estima mejor la SFR y el D4000. Un ensamble aprovecha justamente eso. Según la descomposición de Krogh y Vedelsby (1995), el error de un ensamble es el error medio de sus miembros menos su desacuerdo, de modo que combinar modelos que se equivocan de formas distintas reduce el error por debajo del de cualquiera de ellos. En este avance construimos cuatro ensambles, cubrimos las dos estrategias que pide la actividad y elegimos el modelo final con criterios de desempeño y de uso.

Los **ensambles homogéneos** combinan varias redes de la misma arquitectura entrenadas con semillas distintas. Luz Minutti construyó los de ResNet50 (variantes D y E, tres semillas cada una) y nosotros calculamos aquí, con el mismo criterio de promedio uniforme, sus predicciones sobre las galaxias de validación y de prueba. Los **ensambles heterogéneos** combinan arquitecturas distintas y son los que construimos en este cuaderno, con dos estrategias. La **mezcla convexa** (*blending*) es un promedio ponderado con pesos no negativos que suman uno, aprendidos en un conjunto de reserva (Breiman 1996). El **meta-aprendiz** (*stacking*) es un modelo que recibe las predicciones de todos los miembros y aprende a corregir su promedio (Wolpert 1992). Ambos se entrenan con el conjunto de validación como reserva, porque reentrenar cada red varias veces sobre pliegues del entrenamiento costaría horas de GPU por red; la guía del curso (Singh 2023) describe esta variante como *blending*, y así la llamamos cuando conviene distinguirla.

Como la validación ya sirvió para elegir la época de cada red, y ahora también para entrenar a los combinadores, usamos validación cruzada anidada (Cawley y Talbot 2010): el ciclo interno elige los hiperparámetros del combinador y el externo mide el desempeño en galaxias que no intervinieron en esa elección. El conjunto de prueba se abre una sola vez, al final, con el modelo final ya registrado.

Este cuaderno es el entregable del avance y se apoya en otros dos de la misma carpeta. `Ensambles_Homogeneos_ResNet50_D_E_vf.ipynb`, de Luz Minutti, construye y optimiza los ensambles homogéneos de ResNet50. `Avance5_Entrenamiento_Modelos_Base.ipynb` entrena en GPU los miembros nuevos (DINOv3 y ConvNeXt con redshift, y EfficientNet-B3) y exporta las predicciones de todas las redes que aquí combinamos.

**El interruptor de la pista.** Trabajamos con dos pistas, que se separan con una sola variable. La pista `con_redshift` usa modelos que reciben el redshift espectroscópico como entrada, igual que la variante E de ResNet50. La pista `solo_imagen` usa modelos que solo reciben la imagen, que es la situación de la mayoría de las galaxias a las que queremos extender las estimaciones. Con `PISTA = "con_redshift"` (valor por defecto) el cuaderno construye los ensambles de la primera; al cambiar la variable construye los de la segunda, con exactamente el mismo código.

## 2. Parámetros

| Parámetro | Valor | Significado |
|---|---|---|
| `PISTA` | `con_redshift` | Interruptor de la pista: `con_redshift` o `solo_imagen` |
| `PLIEGUES_EXTERNOS` | 5 | Pliegues de la validación cruzada que mide el desempeño de cada combinador |
| `REPETICIONES` | 3 | Veces que se repite la validación cruzada con otra partición, para medir su variación |
| `PLIEGUES_INTERNOS` | 4 | Pliegues del ciclo interno, donde se eligen los hiperparámetros |
| `PRUEBAS_STACKING` | 40 | Configuraciones que prueba la búsqueda de Optuna del meta-aprendiz |
| `N_REMUESTREOS` | 2000 | Remuestreos *bootstrap* para los intervalos de confianza |
| `SEMILLA_REPRESENTANTE` | 42 | Semilla de la red que representa a cada familia como modelo individual, fijada antes de ver resultados |
| `MARGEN` | 0.02, 0.02, 0.01 | Diferencia de RMSE que se considera despreciable en log M\*, log SFR y D4000 |

Los miembros de cada pista son las familias de modelos del equipo. Cada familia aporta el promedio de sus tres semillas, que es su capa homogénea, y los ensambles heterogéneos combinan un representante de cada arquitectura.

| Pista | Familias que se combinan | Otras familias que se reportan |
|---|---|---|
| `con_redshift` | `resnet50_e`, `dinov3_z`, `convnext_z`, `efficientnet_z` | `resnet50_d` (segundo ensamble homogéneo de Luz) |
| `solo_imagen` | `resnet50_g`, `dinov3_imagen`, `convnext_imagen`, `efficientnet_imagen` | `resnet50_f` |

In [ ]:
import os
variable_entorno = os.environ.get          # las variables MA_* solo se usan en la prueba local reducida

# ------------------------------------------------------------------------------------------------
# INTERRUPTOR DE LA PISTA: "con_redshift" (redshift espectroscópico como entrada) o "solo_imagen".
PISTA = "con_redshift"
# ------------------------------------------------------------------------------------------------
PISTA = variable_entorno("MA_PISTA", PISTA)
PISTAS = {"con_redshift": {"heterogeneo": ["resnet50_e", "dinov3_z", "convnext_z", "efficientnet_z"], "otros": ["resnet50_d"],
                           "redshift_en_meta": True, "descripcion": "con el redshift espectroscópico como entrada"},
          "solo_imagen": {"heterogeneo": ["resnet50_g", "dinov3_imagen", "convnext_imagen", "efficientnet_imagen"], "otros": ["resnet50_f"],
                          "redshift_en_meta": False, "descripcion": "solo con la imagen"}}
if PISTA not in PISTAS:
    raise ValueError(f"PISTA debe ser una de {list(PISTAS)}")
REDSHIFT_EN_META = PISTAS[PISTA]["redshift_en_meta"]       # el redshift solo puede ser entrada del meta-aprendiz si la pista lo usa

PLIEGUES_EXTERNOS = int(variable_entorno("MA_PLIEGUES_EXTERNOS", "5"))
REPETICIONES = int(variable_entorno("MA_REPETICIONES", "3"))
PLIEGUES_INTERNOS = int(variable_entorno("MA_PLIEGUES_INTERNOS", "4"))
PRUEBAS_STACKING = int(variable_entorno("MA_PRUEBAS_STACKING", "40"))
N_REMUESTREOS = int(variable_entorno("MA_REMUESTREOS", "2000"))
SEMILLA_REPRESENTANTE = 42
SEMILLA_ANIDADA = 7
N_TRABAJOS = int(variable_entorno("MA_TRABAJOS", "-1"))
OBJETIVOS = ("log_mstar", "log_sfr", "d4000")
NOMBRES = {"log_mstar": "log M* [dex]", "log_sfr": "log SFR [dex]", "d4000": "D4000"}
CORTO = {"log_mstar": "M*", "log_sfr": "SFR", "d4000": "D4000"}
MARGEN = {"M*": 0.02, "SFR": 0.02, "D4000": 0.01}
print(f"Pista: {PISTA} ({PISTAS[PISTA]['descripcion']})")

## 3. Entorno

Este cuaderno no necesita GPU ni acceso a las imágenes: trabaja con las predicciones que cada red exportó en un formato común (un archivo pequeño por familia y semilla, con las galaxias de validación y de prueba), que están en el repositorio. Descargamos el código del equipo para ligar los resultados a una versión concreta e instalamos Optuna.

In [ ]:
import sys, json, time, subprocess, platform, warnings, hashlib
from pathlib import Path
from datetime import datetime, timezone

EN_COLAB = "google.colab" in sys.modules
URL_REPOSITORIO = "https://github.com/erteck/InferenciaPropiedadesFisicas.git"
CARPETA_REPOSITORIO = Path(variable_entorno("MA_CARPETA_REPOSITORIO", "/content/InferenciaPropiedadesFisicas"))
if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna>=4"], check=True)
    if not CARPETA_REPOSITORIO.exists():
        subprocess.run(["git", "clone", "-q", URL_REPOSITORIO, str(CARPETA_REPOSITORIO)], check=True)
    subprocess.run(["git", "-C", str(CARPETA_REPOSITORIO), "pull", "-q"], check=True)
elif not (CARPETA_REPOSITORIO / "src").exists():
    CARPETA_REPOSITORIO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "modelos_alternativos").exists())
sys.path.insert(0, str(CARPETA_REPOSITORIO))
COMMIT = subprocess.run(["git", "-C", str(CARPETA_REPOSITORIO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, optuna
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from IPython.display import display
from src.modelos_alternativos import ensambles, evaluacion, predicciones

CARPETA_PREDICCIONES = Path(variable_entorno("MA_CARPETA_PREDICCIONES", str(CARPETA_REPOSITORIO / "Avance 5. Modelo final" / "predicciones")))
CARPETA_ENTREGA = CARPETA_PREDICCIONES.parent
pd.set_option("display.precision", 4, "display.max_columns", 40, "display.width", 220)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
print("Python", platform.python_version(), "· commit", COMMIT[:12], "· optuna", optuna.__version__, "·", CARPETA_PREDICCIONES)

## 4. Predicciones de los modelos base

Leemos las predicciones de todas las redes y las alineamos por el identificador de cada galaxia (`mangaid`), nunca por posición. Si alguna red no trae predicción para una galaxia, esa galaxia se descarta para todas, de modo que todos los modelos se comparan sobre exactamente la misma muestra. También se comprueba que las etiquetas y el redshift coincidan entre archivos. La métrica principal usa la desviación de cada propiedad en las galaxias de entrenamiento, que viene en un archivo aparte porque este cuaderno no tiene acceso al catálogo.

In [ ]:
TABLA = predicciones.leer_carpeta(CARPETA_PREDICCIONES)
ESCALA = predicciones.leer_escala(CARPETA_PREDICCIONES)
CONFIGURACION = PISTAS[PISTA]
CANDIDATAS = CONFIGURACION["heterogeneo"] + CONFIGURACION["otros"]
DISPONIBLES = [f for f in CANDIDATAS if f in set(TABLA[TABLA.particion == "Validation"].familia) and f in set(TABLA[TABLA.particion == "Test"].familia)]
FALTAN = [f for f in CANDIDATAS if f not in DISPONIBLES]
if FALTAN:
    print("Aviso: no hay predicciones de validación y de prueba para:", FALTAN, "· se trabaja sin ellas")
FAMILIAS_HETEROGENEO = [f for f in CONFIGURACION["heterogeneo"] if f in DISPONIBLES]
if len(FAMILIAS_HETEROGENEO) < 2:
    raise ValueError("Un ensamble heterogéneo necesita al menos dos arquitecturas con predicciones")

VAL = predicciones.construir_conjunto(TABLA, "Validation", DISPONIBLES)
PRUEBA = predicciones.construir_conjunto(TABLA, "Test", DISPONIBLES)
Y_VAL, Y_PRUEBA, Z_VAL, Z_PRUEBA = VAL.y, PRUEBA.y, VAL.redshift, PRUEBA.redshift
resumen_carga = pd.DataFrame({"semillas": {f: len(VAL.semillas[f]) for f in DISPONIBLES},
                              "galaxias de validación": len(VAL.ids), "galaxias de prueba": len(PRUEBA.ids),
                              "tta": TABLA.groupby("familia").tta.first().reindex(DISPONIBLES).map({True: "sí", False: "no"})})
display(resumen_carga)
print("Desviación de cada propiedad en entrenamiento:", dict(zip(OBJETIVOS, ESCALA.round(4))))

Para el costo de cada modelo juntamos los tiempos que registró el cuaderno de entrenamiento de las redes nuevas. Las redes de ResNet50 se entrenaron en otro equipo (una GPU NVIDIA RTX A4000, más lenta que la A100 de Colab), y sus tiempos de entrenamiento son los que reportó Luz en su cuaderno de ensambles homogéneos: 107.05 minutos en total para las tres redes de la variante E y 109.85 para las de la D. Los tiempos de entrenamiento de las dos máquinas no se pueden comparar directamente, así que anotamos siempre el equipo. El tiempo de uso sí es comparable: el cuaderno de entrenamiento mide los milisegundos por galaxia de todas las arquitecturas en la misma A100, incluida la ResNet50, que se mide con la misma arquitectura y el mismo tamaño de imagen (la duración de una predicción no depende de los valores de los pesos). Cuando un dato no se midió, aparece como «no reportado» y no lo estimamos.

In [ ]:
archivos_tiempos = sorted(CARPETA_PREDICCIONES.glob("tiempos_modelos*.csv"))
TIEMPOS = pd.concat([pd.read_csv(a) for a in archivos_tiempos], ignore_index=True) if archivos_tiempos else pd.DataFrame(
    columns=["familia", "semilla", "hardware", "minutos_entrenamiento", "ms_por_galaxia"])
REPORTADOS_RESNET50 = {"resnet50_e": 107.05, "resnet50_d": 109.85}       # minutos totales de las tres redes (cuaderno de ensambles homogéneos)
for familia, minutos_totales in REPORTADOS_RESNET50.items():
    entrenadas = TIEMPOS[(TIEMPOS.familia == familia) & TIEMPOS.minutos_entrenamiento.notna()]
    if familia in DISPONIBLES and entrenadas.empty:
        filas_nuevas = [{"familia": familia, "semilla": s, "hardware": "NVIDIA RTX A4000", "minutos_entrenamiento": minutos_totales / 3,
                         "ms_por_galaxia": np.nan} for s in VAL.semillas[familia]]
        TIEMPOS = pd.concat([TIEMPOS, pd.DataFrame(filas_nuevas)], ignore_index=True)
primero_valido = lambda x: x.dropna().iloc[0] if x.notna().any() else np.nan
entrenamiento = TIEMPOS[TIEMPOS.minutos_entrenamiento.notna()]                # filas de entrenamiento (las de solo inferencia no cuentan)
COSTO_FAMILIA = pd.DataFrame({"minutos_por_red": entrenamiento.groupby("familia").minutos_entrenamiento.mean(),
                              "hardware": entrenamiento.groupby("familia").hardware.first(),
                              "ms_por_galaxia": TIEMPOS.groupby("familia").ms_por_galaxia.agg(primero_valido)})

def costo_de(familias):
    # Costo de un ensamble que usa todas las semillas de las familias dadas.
    redes = sum(len(VAL.semillas[f]) for f in familias)
    minutos = sum(len(VAL.semillas[f]) * COSTO_FAMILIA.minutos_por_red.get(f, np.nan) for f in familias)
    ms = sum(len(VAL.semillas[f]) * COSTO_FAMILIA.ms_por_galaxia.get(f, np.nan) for f in familias)
    equipos = sorted({h if isinstance(h, str) else "no reportado" for h in (COSTO_FAMILIA.hardware.get(f, np.nan) for f in familias)})
    return {"redes por galaxia": redes, "minutos de entrenamiento": minutos, "ms por galaxia": ms,
            "horas de GPU por millón": ensambles.horas_gpu_por_millon(ms) if np.isfinite(ms) else np.nan, "equipo": " + ".join(equipos)}
display(COSTO_FAMILIA.round(3).fillna("no reportado"))

## 5. Modelos individuales y ensambles homogéneos

Cada familia se representa de dos formas. Como **modelo individual** usamos la red de la semilla 42, la misma que representó a la variante E de ResNet50 en el Avance 4, y la fijamos antes de ver los resultados para no elegir la semilla que mejor salió. Como **ensamble homogéneo** usamos el promedio uniforme de sus tres semillas. Luz Minutti buscó además pesos aprendidos para sus ensambles de ResNet50. En la variante E la búsqueda eligió el propio promedio uniforme, y en la D eligió pesos por propiedad con λ = 0.01, cuya ventaja dentro de la búsqueda fue de una diezmilésima (0.11899 contra 0.11910). Por eso usamos el promedio uniforme en todas las familias: no tiene hiperparámetros y no pierde nada que se distinga del ruido.

La tabla ordena todos los modelos por la métrica principal, el promedio del MSE normalizado entre las tres propiedades (cuanto menor, mejor), y añade el RMSE, el MAE y el R² de cada propiedad. Los valores son de validación; en los individuales y los homogéneos no hay nada que ajustar con esas galaxias, así que son directamente comparables.

In [ ]:
def metricas_largas(nombre, y, prediccion, **extra):
    tabla = evaluacion.tabla_metricas(y, prediccion, OBJETIVOS, ESCALA).set_index("objetivo")
    return {"modelo": nombre, **extra, "criterio": tabla.MSE_normalizado.mean(),
            **{f"RMSE {CORTO[t]}": tabla.loc[t, "RMSE"] for t in OBJETIVOS}, **{f"MAE {CORTO[t]}": tabla.loc[t, "MAE"] for t in OBJETIVOS},
            **{f"R2 {CORTO[t]}": tabla.loc[t, "R2"] for t in OBJETIVOS}}

def indice_representante(familia):
    semillas = VAL.semillas[familia]
    return semillas.index(SEMILLA_REPRESENTANTE) if SEMILLA_REPRESENTANTE in semillas else 0

INDIVIDUAL_VAL = {f: VAL.pred[f][indice_representante(f)] for f in DISPONIBLES}
INDIVIDUAL_PRUEBA = {f: PRUEBA.pred[f][indice_representante(f)] for f in DISPONIBLES}
HOMOGENEO_VAL = {f: VAL.media(f) for f in DISPONIBLES}
HOMOGENEO_PRUEBA = {f: PRUEBA.media(f) for f in DISPONIBLES}

filas = []
for f in DISPONIBLES:
    por_semilla = [ensambles.criterio(Y_VAL, VAL.pred[f][i], ESCALA) for i in range(len(VAL.semillas[f]))]
    filas.append({**metricas_largas(f"{f} · individual (semilla {VAL.semillas[f][indice_representante(f)]})", Y_VAL, INDIVIDUAL_VAL[f], tipo="individual", familia=f),
                  "criterio medio de las semillas": np.mean(por_semilla), "desviación entre semillas": np.std(por_semilla, ddof=1) if len(por_semilla) > 1 else np.nan})
    filas.append(metricas_largas(f"{f} · ensamble homogéneo ({len(VAL.semillas[f])} semillas)", Y_VAL, HOMOGENEO_VAL[f], tipo="homogéneo", familia=f))
BASE = pd.DataFrame(filas).sort_values("criterio").reset_index(drop=True)
display(BASE[["modelo", "criterio", "criterio medio de las semillas", "desviación entre semillas", "RMSE M*", "RMSE SFR", "RMSE D4000", "R2 M*", "R2 SFR", "R2 D4000"]].round(4))
MEJOR_INDIVIDUAL = BASE[BASE.tipo == "individual"].iloc[0].familia
print("Mejor modelo individual (semilla representante):", MEJOR_INDIVIDUAL)

_Se completa con los resultados de la corrida._

## 6. Diversidad de los errores

Antes de combinar medimos cuánto se complementan las familias. Si los residuos de dos modelos están muy correlacionados, cometen los mismos errores y combinarlos aporta poco; si están poco correlacionados, sus errores se compensan. La descomposición de Krogh y Vedelsby (1995) lo cuantifica: para un promedio uniforme de las familias, el error del ensamble es igual al error medio de sus miembros menos su ambigüedad, el desacuerdo medio de los miembros respecto del ensamble, y la ambigüedad es la ganancia que un ensamble puede capturar. Todo se calcula con los promedios de las semillas de cada familia y en unidades de la desviación de entrenamiento, para poder sumar propiedades.

In [ ]:
P_VAL = VAL.medias(FAMILIAS_HETEROGENEO)              # (familias, galaxias, propiedades)
P_PRUEBA = PRUEBA.medias(FAMILIAS_HETEROGENEO)
correlaciones = ensambles.correlacion_residuos(P_VAL, Y_VAL)
minimo_corr = float(min(0.5, np.floor(10 * correlaciones.min()) / 10))
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)
for k, (eje, matriz, t) in enumerate(zip(ejes, correlaciones, OBJETIVOS)):
    imagen = eje.imshow(matriz, vmin=minimo_corr, vmax=1, cmap="viridis")
    eje.set(xticks=range(len(FAMILIAS_HETEROGENEO)), yticks=range(len(FAMILIAS_HETEROGENEO)), title=NOMBRES[t])
    eje.set_xticklabels(FAMILIAS_HETEROGENEO, rotation=45, ha="right", fontsize=8)
    if k == 0:
        eje.set_yticklabels(FAMILIAS_HETEROGENEO, fontsize=8)
    for i in range(len(FAMILIAS_HETEROGENEO)):
        for j in range(len(FAMILIAS_HETEROGENEO)):
            eje.text(j, i, f"{matriz[i, j]:.2f}", ha="center", va="center", fontsize=8, color="w" if matriz[i, j] < (minimo_corr + 1) / 2 else "k")
fig.colorbar(imagen, ax=ejes, shrink=0.75, pad=0.02, label="correlación de los residuos")
plt.suptitle("Correlación de los errores entre arquitecturas en validación (cuanto menor, más se complementan)", y=1.0); plt.show()
AMBIGUEDAD = ensambles.descomposicion_ambiguedad(P_VAL, Y_VAL, ESCALA, nombres_objetivos=[CORTO[t] for t in OBJETIVOS]).set_index("propiedad")
AMBIGUEDAD["ganancia relativa"] = AMBIGUEDAD["ambigüedad"] / AMBIGUEDAD["error medio de los miembros"]
display(AMBIGUEDAD.round(4))

_Se completa con los resultados de la corrida._

## 7. Ensamble heterogéneo 1: mezcla convexa

La mezcla convexa predice cada propiedad como un promedio ponderado de las familias, con pesos no negativos que suman uno, de modo que la predicción nunca queda fuera del intervalo que abarcan sus miembros. Los pesos se obtienen minimizando el error normalizado en la reserva más una penalización que los acerca al promedio uniforme: con una penalización grande se recupera el promedio simple, y sin penalización los pesos siguen solo al error. Esta penalización λ y el alcance de los pesos son los hiperparámetros que optimizamos: con alcance **por propiedad** cada una tiene sus propios pesos (por ejemplo, la masa puede apoyarse en una familia y la SFR en otra), y con alcance **global** las tres comparten los mismos.

| Hiperparámetro | Valores | Qué controla |
|---|---|---|
| Penalización λ | 0, 0.001, 0.01, 0.1, 1, 10 | Cuánto se acercan los pesos al promedio uniforme; la misma rejilla del ensamble homogéneo de Luz |
| Alcance de los pesos | por propiedad, global | Si cada propiedad tiene sus propios pesos |

Evaluamos las doce combinaciones con validación cruzada anidada: en cada pliegue externo, la combinación se elige con los otros pliegues y se mide en el reservado. El resultado es la predicción fuera de pliegue de cada galaxia, que es la que usamos en la comparación.

In [ ]:
CANDIDATOS_H1 = [{"tipo": "mezcla_convexa", "lambda": l, "alcance": a} for a in ("por_objetivo", "global") for l in (0.0, 1e-3, 1e-2, 1e-1, 1.0, 10.0)]
inicio = time.time()
ANIDADA_H1 = ensambles.validacion_anidada(P_VAL, Y_VAL, None, ESCALA, candidatos=CANDIDATOS_H1, pliegues_externos=PLIEGUES_EXTERNOS,
                                          repeticiones=REPETICIONES, pliegues_internos=PLIEGUES_INTERNOS, semilla=SEMILLA_ANIDADA, n_trabajos=N_TRABAJOS)
CONFIG_H1, PRUEBAS_H1 = ensambles.buscar_configuracion(P_VAL, Y_VAL, None, ESCALA, candidatos=CANDIDATOS_H1, pliegues_internos=PLIEGUES_EXTERNOS,
                                                       semilla=SEMILLA_ANIDADA)
PESOS_H1 = ensambles.pesos_mezcla_convexa(P_VAL, Y_VAL, ESCALA, CONFIG_H1["lambda"], CONFIG_H1["alcance"])
print(f"Validación anidada del blending: criterio {ANIDADA_H1['criterio_medio']:.4f} ± {ANIDADA_H1['criterio_desviacion']:.4f} entre repeticiones · {time.time() - inicio:.0f} s")
print("Configuración elegida con toda la reserva:", {k: v for k, v in CONFIG_H1.items() if k != "tipo"})
display(PRUEBAS_H1.head(6).round(5))
fig, eje = plt.subplots(figsize=(7.5, 3.6))
ancho = 0.8 / len(FAMILIAS_HETEROGENEO)
for i, f in enumerate(FAMILIAS_HETEROGENEO):
    eje.bar(np.arange(3) + i * ancho, PESOS_H1[i], ancho * 0.9, label=f)
eje.set(xticks=np.arange(3) + ancho * (len(FAMILIAS_HETEROGENEO) - 1) / 2, xticklabels=[NOMBRES[t] for t in OBJETIVOS], ylabel="peso en la mezcla",
        title="Pesos de la mezcla convexa por propiedad (ajustados con toda la validación)")
eje.axhline(1 / len(FAMILIAS_HETEROGENEO), color="k", ls=":", lw=0.8, label="promedio uniforme"); eje.legend(fontsize=7)
plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

## 8. Ensamble heterogéneo 2: meta-aprendiz

El meta-aprendiz recibe las predicciones de todas las familias para las tres propiedades y aprende a **corregir el promedio uniforme**: en lugar de predecir la propiedad desde cero, predice cuánto hay que sumar al promedio de los miembros. Así, cuando se regulariza con fuerza, el resultado se acerca al promedio y no a una constante, y un meta-aprendiz flexible no puede empeorar mucho a un promedio razonable. Darle las tres propiedades a la vez es una decisión nuestra, no una recomendación de la literatura: las tres están relacionadas (la SFR es la sSFR más la masa) y un error en la masa suele arrastrar el de la SFR. Lo comparamos con la variante que usa solo la propiedad correspondiente.

Optuna (Akiba et al. 2019) elige el tipo de meta-aprendiz y sus hiperparámetros con el muestreo bayesiano TPE, dentro de la validación cruzada anidada.

| Hiperparámetro | Espacio de búsqueda | Qué controla |
|---|---|---|
| Tipo de meta-aprendiz | regresión lineal con pesos no negativos; red elástica con pesos no negativos; Ridge sobre la corrección del promedio; bosque de gradiente poco profundo sobre la corrección | Familia del combinador; los dos primeros se parecen a la mezcla convexa pero con intercepto y sin la restricción de suma uno, lo que les permite corregir la compresión hacia el centro |
| α de la red elástica | 10⁻⁴ a 1 (logarítmica) | Regularización |
| Relación L1 de la red elástica | 0.1 a 0.9 | Mezcla entre selección de miembros y encogimiento |
| α de Ridge | 0.1 a 10⁴ (logarítmica) | Regularización |
| Entradas de Ridge | las tres propiedades; solo la propiedad correspondiente | Si el combinador ve los errores de las otras propiedades |
| Desacuerdo entre familias | sí, no | Si recibe como entrada la desviación entre los miembros, una medida de su incertidumbre |
| Redshift espectroscópico | sí, no (solo en la pista `con_redshift`) | Si recibe el redshift para que los pesos dependan de la distancia |
| Profundidad, tasa de aprendizaje e iteraciones del bosque | 2 a 3, 0.02 a 0.2, 30 a 150 | Flexibilidad del bosque; con 1,261 galaxias esperamos que sobreajuste, y la validación anidada lo comprobará |

In [ ]:
def proponer_stacking(prueba):
    tipo = prueba.suggest_categorical("tipo", ["lineal_positivo", "elastica_positiva", "ridge_correccion", "bosque_residual"])
    usar_z = prueba.suggest_categorical("usar_z", [True, False]) if (REDSHIFT_EN_META and tipo in ("ridge_correccion", "bosque_residual")) else False
    if tipo == "lineal_positivo":
        return {"tipo": tipo}
    if tipo == "elastica_positiva":
        return {"tipo": tipo, "alpha": prueba.suggest_float("alpha_elastica", 1e-4, 1.0, log=True), "l1_ratio": prueba.suggest_float("l1_ratio", 0.1, 0.9)}
    if tipo == "ridge_correccion":
        return {"tipo": tipo, "alpha": prueba.suggest_float("alpha_ridge", 0.1, 1e4, log=True),
                "entradas": prueba.suggest_categorical("entradas", ["todas", "misma_propiedad"]), "usar_z": usar_z,
                "usar_desacuerdo": prueba.suggest_categorical("usar_desacuerdo", [True, False])}
    return {"tipo": tipo, "profundidad": prueba.suggest_int("profundidad", 2, 3), "tasa": prueba.suggest_float("tasa", 0.02, 0.2, log=True),
            "iteraciones": prueba.suggest_int("iteraciones", 30, 150), "usar_z": usar_z, "usar_desacuerdo": prueba.suggest_categorical("usar_desacuerdo_bosque", [True, False])}

Z_META_VAL = Z_VAL if REDSHIFT_EN_META else None
Z_META_PRUEBA = Z_PRUEBA if REDSHIFT_EN_META else None
inicio = time.time()
ANIDADA_H2 = ensambles.validacion_anidada(P_VAL, Y_VAL, Z_META_VAL, ESCALA, proponer=proponer_stacking, pliegues_externos=PLIEGUES_EXTERNOS,
                                          repeticiones=REPETICIONES, pliegues_internos=PLIEGUES_INTERNOS, n_pruebas=PRUEBAS_STACKING,
                                          semilla=SEMILLA_ANIDADA, n_trabajos=N_TRABAJOS)
CONFIG_H2, PRUEBAS_H2 = ensambles.buscar_configuracion(P_VAL, Y_VAL, Z_META_VAL, ESCALA, proponer=proponer_stacking, pliegues_internos=PLIEGUES_EXTERNOS,
                                                       n_pruebas=PRUEBAS_STACKING, semilla=SEMILLA_ANIDADA)
print(f"Validación anidada del meta-aprendiz: criterio {ANIDADA_H2['criterio_medio']:.4f} ± {ANIDADA_H2['criterio_desviacion']:.4f} entre repeticiones · {time.time() - inicio:.0f} s")
print("Configuración elegida con toda la reserva:", {k: (round(v, 5) if isinstance(v, float) else v) for k, v in CONFIG_H2.items()})
print("Tipo de meta-aprendiz elegido en cada pliegue externo:", ANIDADA_H2["configuraciones"].tipo.value_counts().to_dict())
display(PRUEBAS_H2.head(8).round(5))

In [ ]:
mejor_por_tipo = PRUEBAS_H2.groupby("tipo").criterio.min().sort_values()
peor_prueba = PRUEBAS_H2.criterio.max()
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6))
ejes[0].plot(np.arange(1, len(PRUEBAS_H2) + 1), pd.Series(PRUEBAS_H2.sort_index().criterio.to_numpy()).cummin(),
             color="C3", label="mejor hasta el momento")
ejes[0].scatter(np.arange(1, len(PRUEBAS_H2) + 1), PRUEBAS_H2.sort_index().criterio, s=14, color="#264653", label="prueba")
ejes[0].axhline(ensambles.criterio(Y_VAL, P_VAL.mean(0), ESCALA), color="k", ls=":", lw=1, label="promedio uniforme (sin ajustar)")
ejes[0].set(xlabel="prueba", ylabel="criterio en los pliegues internos", title="Avance de la búsqueda"); ejes[0].legend(fontsize=7)
for i, (tipo, g) in enumerate(PRUEBAS_H2.groupby("tipo")):
    ejes[1].scatter(np.full(len(g), i) + np.random.default_rng(0).uniform(-0.12, 0.12, len(g)), g.criterio, s=14, alpha=0.7)
    ejes[1].scatter(i, g.criterio.min(), marker="_", s=500, color="C3")
ejes[1].set_xticks(range(PRUEBAS_H2.tipo.nunique()), sorted(PRUEBAS_H2.tipo.unique()), rotation=20, fontsize=8)
ejes[1].set(ylabel="criterio en los pliegues internos", title="Criterio según el tipo de meta-aprendiz (raya roja: la mejor prueba)")
plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

## 9. Comparación en validación

Reunimos el mejor modelo individual, los ensambles homogéneos y los dos heterogéneos, ordenados por la métrica principal, con otras tres métricas por propiedad y con el costo de cada uno. Para los dos heterogéneos usamos las predicciones fuera de pliegue de la validación anidada, de modo que ninguna galaxia fue predicha por un combinador ajustado con su propia etiqueta; el criterio de la tabla es el promedio de las repeticiones y su desviación indica cuánto cambia con la partición. Los ensambles homogéneos y los individuales no ajustan nada con la validación y se evalúan directamente.

El costo tiene tres partes. Los minutos de entrenamiento suman el de todas las redes del modelo, sin contar las búsquedas de hiperparámetros ni el ajuste del combinador, que dura segundos. Las redes por galaxia y los milisegundos por galaxia (sin TTA, en la A100) miden lo que cuesta usarlo, y de ahí extrapolamos las horas de GPU para un millón de galaxias. Un ensamble necesita ejecutar todas sus redes, así que su costo de uso es la suma de los de sus miembros; las predicciones que combinamos usan TTA de 8 simetrías, que multiplicaría ese costo por ocho si se conserva al desplegar.

In [ ]:
def familias_activas_h1():
    return [f for i, f in enumerate(FAMILIAS_HETEROGENEO) if PESOS_H1[i].max() > 0.01]

def fila_modelo(nombre, tipo, prediccion_oof, familias, obligatorio, criterio_medio=None, desviacion=np.nan, detalle=""):
    costo = costo_de(familias)
    fila = metricas_largas(nombre, Y_VAL, prediccion_oof, tipo=tipo)
    if criterio_medio is not None:
        fila["criterio"] = criterio_medio
    return {**fila, "desviación entre repeticiones": desviacion, **costo, "obligatorio": obligatorio, "detalle": detalle}

def criterios_repeticiones(resultado):
    return resultado["criterio_medio"], resultado["criterio_desviacion"]

FILAS = []
mejor_individual_val = INDIVIDUAL_VAL[MEJOR_INDIVIDUAL]
costo_individual = lambda f: {**costo_de([f]), "redes por galaxia": 1,
                              "minutos de entrenamiento": COSTO_FAMILIA.minutos_por_red.get(f, np.nan),
                              "ms por galaxia": COSTO_FAMILIA.ms_por_galaxia.get(f, np.nan)}
fila = metricas_largas(f"Individual: {MEJOR_INDIVIDUAL} (mejor red individual)", Y_VAL, mejor_individual_val, tipo="individual")
ms = costo_individual(MEJOR_INDIVIDUAL)["ms por galaxia"]
FILAS.append({**fila, "desviación entre repeticiones": np.nan, **costo_individual(MEJOR_INDIVIDUAL),
              "horas de GPU por millón": ensambles.horas_gpu_por_millon(ms) if np.isfinite(ms) else np.nan, "obligatorio": True, "detalle": "una red"})
for f in DISPONIBLES:
    obligatorio = f in ("resnet50_e", "resnet50_d", "resnet50_g", "resnet50_f")
    FILAS.append(fila_modelo(f"Homogéneo: {f} ({len(VAL.semillas[f])} semillas)", "homogéneo", HOMOGENEO_VAL[f], [f], obligatorio,
                             detalle="promedio uniforme de las semillas"))
FILAS.append(fila_modelo("Heterogéneo 1: mezcla convexa", "heterogéneo", ANIDADA_H1["oof"], familias_activas_h1(), True, *criterios_repeticiones(ANIDADA_H1),
                         detalle=f"λ={CONFIG_H1['lambda']}, pesos {CONFIG_H1['alcance'].replace('_', ' ')}"))
FILAS.append(fila_modelo("Heterogéneo 2: meta-aprendiz", "heterogéneo", ANIDADA_H2["oof"], FAMILIAS_HETEROGENEO, True, *criterios_repeticiones(ANIDADA_H2),
                         detalle=f"tipo {CONFIG_H2['tipo']}"))
COMPARACION = pd.DataFrame(FILAS).sort_values("criterio").reset_index(drop=True)
COMPARACION.index += 1; COMPARACION.index.name = "lugar"
display(COMPARACION[["modelo", "criterio", "desviación entre repeticiones", "RMSE M*", "RMSE SFR", "RMSE D4000", "MAE M*", "MAE SFR", "MAE D4000",
                     "R2 M*", "R2 SFR", "R2 D4000"]].round(4).fillna({"desviación entre repeticiones": "no aplica"}))
display(COMPARACION[["modelo", "redes por galaxia", "minutos de entrenamiento", "equipo", "ms por galaxia", "horas de GPU por millón", "detalle"]]
        .round(3).fillna("no reportado"))

In [ ]:
fig, ejes = plt.subplots(1, 4, figsize=(17, 0.9 + 0.5 * len(COMPARACION)), sharey=True, gridspec_kw={"width_ratios": [1.3, 1, 1, 1]})
y_pos = np.arange(len(COMPARACION))[::-1]
paleta = {"individual": "#8d99ae", "homogéneo": "#2a9d8f", "heterogéneo": "#e76f51"}
colores = [paleta[t] for t in COMPARACION.tipo]
ejes[0].barh(y_pos, COMPARACION.criterio, color=colores, xerr=COMPARACION["desviación entre repeticiones"].fillna(0), capsize=3)
ejes[0].set(yticks=y_pos, yticklabels=COMPARACION.modelo, xlabel="MSE normalizado medio (menor es mejor)", title="Métrica principal")
ejes[0].set_xlim(COMPARACION.criterio.min() * 0.8, COMPARACION.criterio.max() * 1.05)
for eje, t in zip(ejes[1:], OBJETIVOS):
    eje.barh(y_pos, COMPARACION[f"RMSE {CORTO[t]}"], color=colores); eje.set(xlabel=f"RMSE {NOMBRES[t]}", title=CORTO[t])
    eje.set_xlim(COMPARACION[f"RMSE {CORTO[t]}"].min() * 0.85, COMPARACION[f"RMSE {CORTO[t]}"].max() * 1.03)
manijas = [plt.Rectangle((0, 0), 1, 1, color=v) for v in paleta.values()]
fig.legend(manijas, paleta.keys(), loc="lower center", ncol=3, frameon=False, fontsize=8)
plt.suptitle(f"Validación ({PISTAS[PISTA]['descripcion']}): modelos ordenados por la métrica principal", y=1.02)
plt.tight_layout(rect=(0, 0.05, 1, 1)); plt.show()

_Se completa con los resultados de la corrida._

## 10. Selección del modelo final

La métrica principal no es el único criterio. El objetivo del proyecto es extender las propiedades espectroscópicas a galaxias que solo tienen imagen, y eso hace que importen el desempeño, la estabilidad, el costo de usar el modelo en catálogos grandes y los datos que necesita al predecir. Para que la decisión no premie una mejora que no se distingue del ruido a cambio de nueve redes más por galaxia, aplicamos la **regla de un error estándar** que usamos en la fase de ingeniería del Avance 3 (Hastie et al. 2009, §7.10): entre los modelos que no se distinguen del mejor en más de un error estándar, elegimos el más simple, que medimos por el número de redes que hay que ejecutar para cada galaxia.

Adaptamos la regla a que todos los modelos predicen las mismas galaxias. El error estándar que usamos es el de la **diferencia pareada** de la pérdida por galaxia entre cada modelo y el mejor. La versión sin parear mide la dispersión de la pérdida entre galaxias, que crece con las galaxias difíciles para todos los modelos (en la SFR, las galaxias apagadas); cuando los errores de los modelos están correlacionados, como esperamos aquí, esa dispersión es común a todos y puede declarar empatados a modelos que la comparación pareada distingue. Las pérdidas de los ensambles heterogéneos son las de sus predicciones fuera de pliegue, promediadas entre las repeticiones de la validación cruzada. Fijamos esta regla antes de tener las predicciones reales, decidimos antes de abrir el conjunto de prueba y registramos la elección con una huella de sus predicciones de validación: si el cuaderno se volviera a ejecutar con otro modelo elegido, el registro lo detectaría.

In [ ]:
def prediccion_validacion(nombre):
    fila = COMPARACION[COMPARACION.modelo == nombre].iloc[0]
    if nombre.startswith("Heterogéneo 1"):
        return ANIDADA_H1["oof"]
    if nombre.startswith("Heterogéneo 2"):
        return ANIDADA_H2["oof"]
    if nombre.startswith("Individual"):
        return mejor_individual_val
    return HOMOGENEO_VAL[nombre.split(": ")[1].split(" (")[0]]

def perdida_validacion(nombre):
    # Pérdida por galaxia; en los heterogéneos, promedio de las repeticiones de la validación anidada.
    if nombre.startswith("Heterogéneo"):
        resultado = ANIDADA_H1 if nombre.startswith("Heterogéneo 1") else ANIDADA_H2
        return np.mean([ensambles.perdida_por_galaxia(Y_VAL, o, ESCALA) for o in resultado["oof_por_repeticion"]], axis=0)
    return ensambles.perdida_por_galaxia(Y_VAL, prediccion_validacion(nombre), ESCALA)

PERDIDAS = {n: perdida_validacion(n) for n in COMPARACION.modelo}
COMPLEJIDAD = dict(zip(COMPARACION.modelo, COMPARACION["redes por galaxia"]))
ELEGIDO, TABLA_REGLA, DETALLE_REGLA = ensambles.regla_un_error_estandar(PERDIDAS, COMPLEJIDAD)
display(TABLA_REGLA.round(4))
print(f"Mejor criterio: {DETALLE_REGLA['mejor']}\nModelo elegido por la regla: {ELEGIDO}\n"
      f"(Como referencia, el error estándar sin parear del mejor sería {DETALLE_REGLA['error_estandar_sin_parear']:.4f}.)")

In [ ]:
NECESITA_REDSHIFT = "sí, espectroscópico" if PISTA == "con_redshift" else "no"
NEGOCIO = COMPARACION[["modelo", "criterio", "desviación entre repeticiones", "redes por galaxia", "horas de GPU por millón"]].copy()
NEGOCIO["entradas al predecir"] = ["imagen + redshift" if PISTA == "con_redshift" else "solo imagen"] * len(NEGOCIO)
NEGOCIO["diferencia con el mejor"] = NEGOCIO.modelo.map(TABLA_REGLA["diferencia con el mejor"])
NEGOCIO["error estándar de la diferencia"] = NEGOCIO.modelo.map(TABLA_REGLA["error estándar de la diferencia"])
NEGOCIO["dentro de 1 error estándar"] = NEGOCIO.modelo.map(TABLA_REGLA["dentro de 1 error estándar"])
display(NEGOCIO.round(4).fillna({"desviación entre repeticiones": "no aplica"}).fillna("no reportado"))

_Se completa con los resultados de la corrida._

## 11. Evaluación en el conjunto de prueba

Con el modelo ya elegido y registrado, abrimos el conjunto de prueba **una sola vez**. Los combinadores heterogéneos se ajustan con toda la validación y se aplican sin cambios a las predicciones de prueba de los miembros. Además del modelo final, reportamos los demás modelos de la comparación solo como contexto: sus resultados no pueden cambiar la elección, que ya está registrada.

Como referencia externa incluimos a Katachi (Alfonzo et al. 2024). Sus cifras publicadas (RMSE de 0.22, 0.75 y 0.28 en su conjunto de prueba de SDSS) no se pueden comparar galaxia por galaxia con las nuestras, así que las mostramos como referencia; sus pesos publicados evaluados sobre nuestras galaxias de prueba y la cadena de ResNet50 que reentrenamos con imágenes de DESI en el Avance 3 (media de tres semillas) sí son comparables.

**Independencia de la prueba.** _Pendiente: confirmar con Luz Minutti si el conjunto de prueba se consultó en alguna fase del entrenamiento o de la selección de las variantes de ResNet50 y, según su respuesta, declarar aquí si la evaluación de los modelos que incluyen a esas redes es independiente o una limitación._

In [ ]:
RUTA_REGISTRO = CARPETA_ENTREGA / f"registro_eleccion_{PISTA}.json"
REGISTRO = ensambles.registrar_eleccion(RUTA_REGISTRO, {"pista": PISTA, "modelo": ELEGIDO, "commit": COMMIT,
                                                        "criterio_validacion": float(COMPARACION.set_index("modelo").loc[ELEGIDO, "criterio"])},
                                        prediccion_validacion(ELEGIDO))

def ajustar_final(nombre):
    # Predicciones de prueba del modelo con su configuración final (ajustada con toda la validación).
    if nombre.startswith("Heterogéneo 1"):
        return ensambles.aplicar_pesos(P_PRUEBA, PESOS_H1)
    if nombre.startswith("Heterogéneo 2"):
        return ensambles.ajustar_predecir(CONFIG_H2, P_VAL, Y_VAL, Z_META_VAL, P_PRUEBA, Z_META_PRUEBA, ESCALA)
    if nombre.startswith("Individual"):
        return INDIVIDUAL_PRUEBA[MEJOR_INDIVIDUAL]
    return HOMOGENEO_PRUEBA[nombre.split(": ")[1].split(" (")[0]]

PRED_PRUEBA = {n: ajustar_final(n) for n in COMPARACION.modelo}
filas = [{**metricas_largas(n, Y_PRUEBA, PRED_PRUEBA[n], tipo=COMPARACION.set_index("modelo").loc[n, "tipo"]), "elegido": n == ELEGIDO} for n in COMPARACION.modelo]
TABLA_PRUEBA = pd.DataFrame(filas).sort_values("criterio").reset_index(drop=True)
display(TABLA_PRUEBA[["modelo", "elegido", "criterio", "RMSE M*", "RMSE SFR", "RMSE D4000", "MAE M*", "MAE SFR", "MAE D4000", "R2 M*", "R2 SFR", "R2 D4000"]].round(4))

REFERENCIAS_KATACHI = pd.DataFrame({"RMSE M*": [0.22, 0.222, 0.223], "RMSE SFR": [0.75, 0.668, 0.653], "RMSE D4000": [0.28, 0.100, 0.073]},
                                   index=["Katachi, artículo (SDSS, otra muestra; no pareable)", "Katachi, pesos publicados en nuestras galaxias de prueba",
                                          "Katachi reentrenado con DESI (Avance 3, media de 3 semillas)"])
display(REFERENCIAS_KATACHI)

_Se completa con los resultados de la corrida._

Para saber si el modelo final es mejor que los demás con una diferencia que no se explique por el azar de la muestra, comparamos con un *bootstrap* pareado: remuestreamos las galaxias de prueba y calculamos en cada remuestreo la diferencia del criterio entre dos modelos. Si el intervalo del 95 % de esa diferencia excluye el cero, la diferencia es significativa.

In [ ]:
FINAL_PRUEBA = PRED_PRUEBA[ELEGIDO]
filas = []
for nombre in COMPARACION.modelo:
    if nombre == ELEGIDO:
        continue
    d, lo, hi = ensambles.diferencia_pareada_criterio(Y_PRUEBA, FINAL_PRUEBA, PRED_PRUEBA[nombre], ESCALA, N_REMUESTREOS)
    filas.append({"comparado con": nombre, "Δ criterio (final − otro)": d, "IC 95 %": f"[{lo:+.4f}, {hi:+.4f}]",
                  "veredicto": "el final es mejor" if hi < 0 else ("el final es peor" if lo > 0 else "sin diferencia significativa")})
display(pd.DataFrame(filas).set_index("comparado con").round(4))

_Se completa con los resultados de la corrida._

## 12. Gráficos del modelo final

Como nuestro problema es de regresión, las curvas ROC y las matrices de confusión no aplican; usamos los gráficos propios de la regresión. Para el modelo final y sobre las galaxias de prueba, que ningún paso del proceso había visto, mostramos la concordancia entre predicción y etiqueta, la distribución de los residuos, cómo cambia el error con el valor de la propiedad y con el redshift, qué aporta cada familia al ensamble, si el desacuerdo entre familias sirve como señal de incertidumbre y cuánto se gana al sumar miembros.

In [ ]:
NOMBRE_FINAL = ELEGIDO
COMPARADO = f"Individual: {MEJOR_INDIVIDUAL} (mejor red individual)"
fig, ejes = plt.subplots(1, 3, figsize=(14, 4.4))
for k, (eje, t) in enumerate(zip(ejes, OBJETIVOS)):
    y, p = Y_PRUEBA[:, k], FINAL_PRUEBA[:, k]
    lim = np.percentile(np.r_[y, p], [0.5, 99.5])
    eje.hexbin(y, p, gridsize=45, cmap="viridis", mincnt=1, bins="log", extent=(*lim, *lim)); eje.plot(lim, lim, "k-", lw=0.8)
    m = evaluacion.metricas(y, p)
    eje.text(0.03, 0.97, f"RMSE {m['RMSE']:.3f}\nR² {m['R2']:.3f}\nsesgo {m['sesgo']:+.3f}\npendiente {ensambles.pendiente_de_calibracion(y, p):.2f}",
             transform=eje.transAxes, va="top", fontsize=8, bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, edgecolor="0.7"))
    eje.set(xlim=lim, ylim=lim, xlabel=f"{NOMBRES[t]} (etiqueta)", ylabel="predicción")
plt.suptitle(f"Prueba: predicción frente a etiqueta · {NOMBRE_FINAL}", y=1.02); plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 3.8))
for k, (eje, t) in enumerate(zip(ejes, OBJETIVOS)):
    for nombre, color, etiqueta in ((COMPARADO, "0.55", "mejor red individual"), (NOMBRE_FINAL, "#e76f51", "modelo final")):
        r = PRED_PRUEBA[nombre][:, k] - Y_PRUEBA[:, k]
        eje.hist(r, bins=60, density=True, histtype="step", lw=1.6, color=color, label=f"{etiqueta}: sesgo {r.mean():+.3f}, σ_NMAD {evaluacion.metricas(Y_PRUEBA[:, k], PRED_PRUEBA[nombre][:, k])['sigma_NMAD']:.3f}")
    eje.axvline(0, color="k", lw=0.6); eje.set(xlabel=f"residuo de {NOMBRES[t]} (predicción − etiqueta)", ylabel="densidad", title=CORTO[t]); eje.legend(fontsize=7)
plt.suptitle("Prueba: distribución de los residuos", y=1.02); plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

In [ ]:
def mediana_por_intervalos(x, residuo, n=10):
    n = int(max(2, min(n, len(x) // 12)))
    bordes = np.quantile(x, np.linspace(0, 1, n + 1)); centro, med, p16, p84 = [], [], [], []
    for a, b in zip(bordes[:-1], bordes[1:]):
        s = (x >= a) & (x <= b)
        if s.sum() >= 4:
            centro.append(np.median(x[s])); med.append(np.median(residuo[s])); p16.append(np.percentile(residuo[s], 16)); p84.append(np.percentile(residuo[s], 84))
    return map(np.array, (centro, med, p16, p84))

fig, ejes = plt.subplots(2, 3, figsize=(14, 7))
for k, t in enumerate(OBJETIVOS):
    for fila, (x, nombre_x) in enumerate(((Y_PRUEBA[:, k], f"{NOMBRES[t]} (etiqueta)"), (Z_PRUEBA, "redshift medido"))):
        for nombre, color, etiqueta in ((COMPARADO, "0.55", "mejor red individual"), (NOMBRE_FINAL, "#e76f51", "modelo final")):
            cx, med, p16, p84 = mediana_por_intervalos(x, PRED_PRUEBA[nombre][:, k] - Y_PRUEBA[:, k])
            ejes[fila, k].plot(cx, med, "o-", ms=3, color=color, label=etiqueta); ejes[fila, k].fill_between(cx, p16, p84, color=color, alpha=0.12)
        ejes[fila, k].axhline(0, color="k", lw=0.6); ejes[fila, k].set(xlabel=nombre_x, ylabel="mediana del residuo")
    ejes[0, k].set_title(NOMBRES[t])
ejes[0, 0].legend(fontsize=7)
plt.suptitle("Prueba: residuo según el valor de la propiedad (arriba) y según el redshift (abajo); franja: 68 % central", y=1.0)
plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

In [ ]:
# Aporte de cada familia al modelo final: se vuelve a ajustar el combinador sin ella y se mide cuánto empeora el criterio en validación (5 pliegues).
if ELEGIDO.startswith("Heterogéneo 2"):
    CONFIG_FINAL = CONFIG_H2
elif ELEGIDO.startswith("Heterogéneo 1"):
    CONFIG_FINAL = {"tipo": "mezcla_convexa", "lambda": CONFIG_H1["lambda"], "alcance": CONFIG_H1["alcance"]}
else:
    CONFIG_FINAL = None
if CONFIG_FINAL is not None:
    _, criterio_completo = ensambles.validar_configuracion(CONFIG_FINAL, P_VAL, Y_VAL, Z_META_VAL, ESCALA, PLIEGUES_EXTERNOS, SEMILLA_ANIDADA)
    filas = []
    for i, f in enumerate(FAMILIAS_HETEROGENEO):
        resto = [j for j in range(len(FAMILIAS_HETEROGENEO)) if j != i]
        oof, criterio_sin = ensambles.validar_configuracion(CONFIG_FINAL, P_VAL[resto], Y_VAL, Z_META_VAL, ESCALA, PLIEGUES_EXTERNOS, SEMILLA_ANIDADA)
        por_propiedad = ensambles.criterio_por_propiedad(Y_VAL, oof, ESCALA)
        filas.append({"familia": f, **{CORTO[t]: por_propiedad[k] for k, t in enumerate(OBJETIVOS)}, "criterio sin la familia": criterio_sin,
                      "aumento del criterio": criterio_sin - criterio_completo})
    ABLACION = pd.DataFrame(filas).set_index("familia")
    referencia_propiedad = ensambles.criterio_por_propiedad(Y_VAL, ensambles.validar_configuracion(CONFIG_FINAL, P_VAL, Y_VAL, Z_META_VAL, ESCALA, PLIEGUES_EXTERNOS, SEMILLA_ANIDADA)[0], ESCALA)
    fig, ejes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1.4, 1]})
    ancho = 0.8 / len(FAMILIAS_HETEROGENEO)
    for i, f in enumerate(FAMILIAS_HETEROGENEO):
        ejes[0].bar(np.arange(3) + i * ancho, [ABLACION.loc[f, CORTO[t]] - referencia_propiedad[k] for k, t in enumerate(OBJETIVOS)], ancho * 0.9, label=f)
    ejes[0].axhline(0, color="k", lw=0.6)
    ejes[0].set(xticks=np.arange(3) + ancho * (len(FAMILIAS_HETEROGENEO) - 1) / 2, xticklabels=[NOMBRES[t] for t in OBJETIVOS],
                ylabel="aumento del MSE normalizado al quitar la familia", title="Aporte de cada familia por propiedad"); ejes[0].legend(fontsize=7)
    if CONFIG_FINAL["tipo"] == "mezcla_convexa":
        imagen = ejes[1].imshow(PESOS_H1.T, cmap="viridis", vmin=0, vmax=1, aspect="auto")
        ejes[1].set(xticks=range(len(FAMILIAS_HETEROGENEO)), yticks=range(3), yticklabels=[CORTO[t] for t in OBJETIVOS], title="Pesos de la mezcla")
        ejes[1].set_xticklabels(FAMILIAS_HETEROGENEO, rotation=45, ha="right", fontsize=7)
        for i in range(len(FAMILIAS_HETEROGENEO)):
            for j in range(3):
                ejes[1].text(i, j, f"{PESOS_H1[i, j]:.2f}", ha="center", va="center", fontsize=8, color="w" if PESOS_H1[i, j] < 0.5 else "k")
    else:
        ejes[1].barh(range(len(FAMILIAS_HETEROGENEO)), ABLACION["aumento del criterio"], color="#e76f51")
        ejes[1].set(yticks=range(len(FAMILIAS_HETEROGENEO)), yticklabels=FAMILIAS_HETEROGENEO, xlabel="aumento del criterio global", title="Aporte global")
    plt.suptitle("Qué aporta cada arquitectura al modelo final (validación, sin la familia)", y=1.02); plt.tight_layout(); plt.show()
    display(ABLACION.round(4))
else:
    print("El modelo final no es un ensamble heterogéneo: no hay aportes por familia que mostrar.")

_Se completa con los resultados de la corrida._

In [ ]:
# Desacuerdo entre familias como señal de incertidumbre: ¿los casos en que los miembros discrepan son los que más fallan?
DESACUERDO = ensambles.desacuerdo_ponderado(P_PRUEBA, PESOS_H1 if ELEGIDO.startswith("Heterogéneo 1") else None)
ERROR_ABSOLUTO = np.abs(FINAL_PRUEBA - Y_PRUEBA)
fig, ejes = plt.subplots(1, 3, figsize=(14, 3.9))
for k, (eje, t) in enumerate(zip(ejes, OBJETIVOS)):
    cuantiles = np.quantile(DESACUERDO[:, k], np.linspace(0, 1, 6))
    grupo = np.clip(np.searchsorted(cuantiles, DESACUERDO[:, k], side="right") - 1, 0, 4)
    medianas = [np.median(ERROR_ABSOLUTO[grupo == g, k]) for g in range(5)]
    p84 = [np.percentile(ERROR_ABSOLUTO[grupo == g, k], 84) for g in range(5)]
    rho = spearmanr(DESACUERDO[:, k], ERROR_ABSOLUTO[:, k]).statistic
    eje.hexbin(DESACUERDO[:, k], ERROR_ABSOLUTO[:, k], gridsize=35, cmap="Greys", mincnt=1, bins="log")
    centros = [np.median(DESACUERDO[grupo == g, k]) for g in range(5)]
    eje.plot(centros, medianas, "o-", color="#e76f51", label="mediana del error por quintil"); eje.plot(centros, p84, "s--", color="#264653", label="percentil 84")
    eje.set(xlabel="desacuerdo entre familias", ylabel="error absoluto", title=f"{NOMBRES[t]} · ρ de Spearman = {rho:.2f}"); eje.legend(fontsize=7)
plt.suptitle("Prueba: ¿sirve el desacuerdo entre familias como señal de incertidumbre?", y=1.02); plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

In [ ]:
# Ganancia al sumar miembros: se agregan las familias en el orden de su aporte y se mide el criterio de validación (fuera de pliegue, configuración final fija).
if CONFIG_FINAL is not None:
    orden = list(ABLACION.sort_values("aumento del criterio", ascending=False).index)
    posiciones = [FAMILIAS_HETEROGENEO.index(f) for f in orden]
    curva = [ensambles.validar_configuracion(CONFIG_FINAL, P_VAL[posiciones[:n]], Y_VAL, Z_META_VAL, ESCALA, PLIEGUES_EXTERNOS, SEMILLA_ANIDADA)[1]
             for n in range(1, len(orden) + 1)]
    fig, eje = plt.subplots(figsize=(7, 3.6))
    eje.plot(range(1, len(orden) + 1), curva, "o-", color="#e76f51")
    for n, (f, v) in enumerate(zip(orden, curva), 1):
        eje.annotate(f"+ {f}" if n > 1 else f, (n, v), textcoords="offset points", xytext=(4, 6), fontsize=7)
    eje.set(xticks=range(1, len(orden) + 1), xlabel="familias combinadas (de mayor a menor aporte)", ylabel="criterio en validación (menor es mejor)",
            title="Rendimientos decrecientes al sumar arquitecturas")
    plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

## 13. Conclusiones

_Se completa con los resultados de la corrida._

## 14. Referencias

Akiba, T., Sano, S., Yanase, T., Ohta, T. y Koyama, M. (2019). Optuna: A next-generation hyperparameter optimization framework. *Proceedings of the 25th ACM SIGKDD*, 2623–2631.

Alfonzo, J. P., Iyer, K. G., Akiyama, M., et al. (2024). Decoding the imprints of past star formation on present-day morphology in galaxies with interpretable CNNs. *The Astrophysical Journal*, 967, 152.

Breiman, L. (1996). Stacked regressions. *Machine Learning*, 24(1), 49–64.

Cawley, G. C. y Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. *Journal of Machine Learning Research*, 11, 2079–2107.

Hastie, T., Tibshirani, R. y Friedman, J. (2009). *The Elements of Statistical Learning* (2.ª ed.). Springer. Sección 7.10.

Krogh, A. y Vedelsby, J. (1995). Neural network ensembles, cross validation, and active learning. *Advances in Neural Information Processing Systems 7*, 231–238.

Singh, A. (2023, 14 de julio). Comprehensive guide to ensemble learning (with Python codes). *Analytics Vidhya*. https://www.analyticsvidhya.com/blog/2018/06/comprehensive-guide-for-ensemble-models/

Studer, S., Bui, T. B., Drescher, C., et al. (2021). Towards CRISP-ML(Q): A machine learning process model with quality assurance methodology. *Machine Learning and Knowledge Extraction*, 3, 392–413.

VanderPlas, J. (2022). *Python Data Science Handbook* (2.ª ed.). O'Reilly Media. Capítulo 39, sección *Thinking about model validation*.

Wolpert, D. H. (1992). Stacked generalization. *Neural Networks*, 5(2), 241–259.

Zhong, J., Deng, Z., Li, X., Wang, L., Yang, H., Li, H. y Zhao, X. (2024). Estimation of stellar mass and star formation rate based on galaxy images. *Monthly Notices of the Royal Astronomical Society*, 531(1), 2011–2027.

# Declaración de uso de IA y contribución del equipo

### Uso de herramientas de IA

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (celdas o secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude [Modelo de lenguaje grande]. https://claude.ai/ | Escribir el código de los cuadernos de entrenamiento y de ensambles, con sus pruebas unitarias, y redactar los análisis a partir de los resultados | Todo este cuaderno y el de entrenamiento de los modelos base | Ejecutamos los entrenamientos y los ensambles, comparamos cada cifra con las salidas de los cuadernos, corregimos el código y los textos, y validamos cada interpretación |
| OpenAI. (2026). ChatGPT (GPT-6) [Modelo de lenguaje grande]. https://chatgpt.com/ | Estructurar el código y revisar el procedimiento de comparación en los modelos 1, 2, 5 y 6 y en los ensambles homogéneos | Cuadernos de los modelos 1, 2, 5 y 6 y de los ensambles homogéneos, de los que este cuaderno toma las predicciones | Revisamos las salidas antes de interpretarlas y adaptamos el código a las variables astronómicas, con la orientación de los asesores |

### Contribución individual

| Integrante | Contribución al avance |
|---|---|
| Luz Copelia Minutti Pérez | Desarrolló y ejecutó los modelos 1 (ResNet50 multitarea), 2 (SVR), 5 (Ridge) y 6 (KNN), el protocolo común de comparación y los ensambles homogéneos de ResNet50, y exportó sus predicciones. |
| Erick Alberto Bustos Cruz | Desarrolló y ejecutó los modelos 3 (ConvNeXt-Tiny) y 4 (DINOv3 con LoRA), entrenó los modelos base con redshift y EfficientNet-B3, construyó los ensambles heterogéneos y redactó la comparación, la selección y la evaluación del modelo final. |
| José Florencio Maguey Peralta | _Pendiente: el integrante debe completar su contribución._ |

### Decisiones principales

<small><b>Decisión 1:</b> Usar la validación como conjunto de reserva de los ensambles heterogéneos, con validación cruzada anidada</small>
- <small>Justificación: reentrenar cada red varias veces sobre pliegues del entrenamiento costaba horas de GPU por red, y la validación cruzada anidada permite medir el desempeño de los combinadores sin que las galaxias evaluadas hayan intervenido en su ajuste.</small>
- <small>Alternativa descartada y motivo: stacking con predicciones por pliegues sobre todo el entrenamiento, porque exigía entrenar de nuevo cada red varias veces (unas 2 horas con DINOv3 y unas 3 con ResNet50).</small>

<small><b>Decisión 2:</b> Construir los ensambles en dos pistas, con y sin el redshift espectroscópico, controladas por una sola variable</small>
- <small>Justificación: el redshift espectroscópico mejora mucho la masa, pero no existe para la mayoría de las galaxias que solo tienen imagen, y el uso final del modelo determina cuál pista conviene.</small>
- <small>Alternativa descartada y motivo: elegir una sola pista desde el inicio, porque habría ocultado cuánto se pierde al no conocer la distancia.</small>

<small><b>Decisión 3:</b> Elegir el modelo final con la regla de un error estándar, y que el meta-aprendiz corrija el promedio uniforme</small>
- <small>Justificación: evita pagar el costo de ejecutar muchas redes por una mejora que no se distingue del ruido, y mantiene al meta-aprendiz cerca de un promedio razonable aunque haya solo 1,261 galaxias de reserva.</small>
- <small>Alternativa descartada y motivo: elegir el modelo con el menor criterio sin más, y un meta-aprendiz libre sobre las predicciones, porque puede sobreajustar con tan pocas galaxias.</small>